# Análisis Exploratorio de Datos (EDA) - Señales EMG (NinaPro DB2)

Este notebook ilustra el proceso de **Carga, Inspección, Filtrado Digital y Ventaneo Temporal** de las señales electromiográficas (EMG) capturadas desde los músculos del antebrazo.

### Objetivos:
1. Visualizar la señal EMG cruda en el dominio del tiempo.
2. Analizar el espectro de frecuencia original para identificar interferencias (artefactos de movimiento y ruido de red de 50/60 Hz).
3. Aplicar filtros paso banda Butterworth ($20-450 \text{ Hz}$) y Notch ($50 \text{ Hz}$) para limpiar la señal.
4. Generar la matriz de características temporales por ventana.

In [ ]:
import sys
import os
import yaml
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

# Agregar el directorio raíz del proyecto al path de Python
sys.path.append(os.path.abspath(".."))

from src.signal_processing.filters import EMGFilter
from src.signal_processing.feature_extraction import extract_time_features, window_signal

# Estilo visual para los gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Entorno configurado correctamente.")

In [ ]:
# Cargar configuración global
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_path = "../data/raw/S1_E1_A1.mat"

# Verificar si el archivo existe o generarlo si no está presente
if not os.path.exists(data_path):
    from data.download_data import generate_synthetic_ninapro
    generate_synthetic_ninapro(data_path, config['dataset']['num_channels'], config['dataset']['sample_rate'])

# Cargar dataset
mat = scipy.io.loadmat(data_path)
emg_raw = mat['emg']             # Forma: [muestras, 12 canales]
restimulus = mat['restimulus'].flatten() # Clase/Gesto asignado a cada muestra

fs = config['dataset']['sample_rate']
time_axis = np.arange(emg_raw.shape[0]) / fs

print(f"Canales EMG: {emg_raw.shape[1]}")
print(f"Duración total: {emg_raw.shape[0] / fs:.2f} segundos ({emg_raw.shape[0]} muestras)")
print(f"Gestos registrados en la muestra: {np.unique(restimulus)}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# Visualizar 3 canales representativos
ax1.plot(time_axis, emg_raw[:, 0], label="Canal 1 (Extensor del antebrazo)", alpha=0.8, color="#2b5c8f")
ax1.plot(time_axis, emg_raw[:, 4], label="Canal 5 (Flexor del antebrazo)", alpha=0.7, color="#d95f02")
ax1.set_ylabel("Amplitud (mV)")
ax1.set_title("Señales EMG Crudas (Raw Signals) - Muestras sin Filtrar", fontsize=14, fontweight='bold')
ax1.legend(loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.6)

# Visualizar el estímulo / clase
ax2.plot(time_axis, restimulus, color="#7570b3", linewidth=2)
ax2.set_xlabel("Tiempo (segundos)")
ax2.set_ylabel("ID de Gesto")
ax2.set_yticks(np.unique(restimulus))
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Aplicar el filtro diseñado en src/signal_processing/filters.py
emg_filter = EMGFilter(
    sample_rate=fs,
    lowcut=config['processing']['lowcut'],
    highcut=config['processing']['highcut'],
    notch_freq=config['processing']['notch_freq']
)

emg_filtered = emg_filter.process(emg_raw)

# Calcular Densidad Espectral de Potencia (PSD) mediante método Welch
f_raw, psd_raw = welch(emg_raw[:, 0], fs=fs, nperseg=fs)
f_filt, psd_filt = welch(emg_filtered[:, 0], fs=fs, nperseg=fs)

# Visualizar PSD
plt.figure(figsize=(12, 5))
plt.semilogy(f_raw, psd_raw, label="Señal Original (Raw)", color="#e41a1c", alpha=0.7)
plt.semilogy(f_filt, psd_filt, label="Señal Filtrada (Butterworth 20-450Hz + Notch 50Hz)", color="#377eb8", linewidth=2)
plt.axvline(20, color='gray', linestyle=':', label='Corte Inferior (20 Hz)')
plt.axvline(450, color='gray', linestyle=':', label='Corte Superior (450 Hz)')
plt.axvline(50, color='green', linestyle='--', label='Filtro Notch (50 Hz)')

plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Densidad Espectral de Potencia (V²/Hz)")
plt.title("Efecto del Filtrado Biomédico en el Espectro de Frecuencia", fontsize=13, fontweight='bold')
plt.xlim(0, 600)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
win_samples = int((config['dataset']['window_size_ms'] / 1000.0) * fs)
step_samples = int(((config['dataset']['window_size_ms'] - config['dataset']['overlap_ms']) / 1000.0) * fs)

# Generar ventanas
X_windows, y_labels = window_signal(emg_filtered, restimulus, win_samples, step_samples)

print(f"Forma de los datos ventaneados (X): {X_windows.shape}") 
# [N_ventanas, muestras_por_ventana, 12_canales]

# Extraer características clasicas por ventana para analisis
features_list = [extract_time_features(win) for win in X_windows]
features_matrix = np.array(features_list)

print(f"Forma de la matriz de características extraídas: {features_matrix.shape}")

In [ ]:
# Calcular RMS promedio por canal en las ventanas activas
rms_per_channel = np.array([np.sqrt(np.mean(win**2, axis=0)) for win in X_windows])

plt.figure(figsize=(10, 8))
corr_matrix = np.corrcoef(rms_per_channel.T)

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="YlGnBu",
    xticklabels=[f"CH{i+1}" for i in range(12)],
    yticklabels=[f"CH{i+1}" for i in range(12)]
)
plt.title("Matriz de Correlación EMG entre Canales del Antebrazo (Amplitud RMS)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusiones del Análisis
1. **Atenuación de Artefactos:** El filtro de banda elimina eficazmente las interferencias de baja frecuencia (< 20 Hz) producidas por los movimientos del cable/electrodo.
2. **Eliminación de Interferencia:** El filtro Notch corta la espiga de $50 \text{ Hz}$ de la red eléctrica.
3. **Formato para Deep Learning:** Las ventanas resultantes de dimensión `[Ventanas, 500 muestras, 12 canales]` están listas para alimentar directamente a la arquitectura **Conv1D-LSTM** definida en `src/models/cnn1d_lstm.py`.